# Notebook 2: Synthetic Data Generation für Photonic Chip Benchmarks

## Ziel
Realistische synthetische Daten für Matrix-Multiplikationen unter photonic-typischen Störungen erzeugen.

In [6]:
# Zelle 1: Imports
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliotheken geladen")

✅ Bibliotheken geladen


In [7]:
# Zelle 2: Kern-Funktion
def photonic_matrix_mult(A: np.ndarray, B: np.ndarray, 
                        noise_level: float = 0.01,
                        drift_factor: float = 0.005,
                        crosstalk_factor: float = 0.003,
                        bit_precision: int = 8,
                        seed: int = 42) -> dict:
    
    np.random.seed(seed)
    ideal = A @ B
    
    # Noise-Komponenten
    gaussian_noise = np.random.normal(0, noise_level * np.abs(ideal), ideal.shape)
    t = np.linspace(0, 10, ideal.shape[0])
    phase_drift = drift_factor * np.sin(t[:, None]) * ideal
    crosstalk = crosstalk_factor * np.random.normal(0, 1, ideal.shape) * A.mean(axis=1)[:, None]
    
    noisy = ideal + gaussian_noise + phase_drift + crosstalk
    
    # Quantisierung
    scale = 2 ** (bit_precision - 1)
    quantized = np.round(noisy * scale) / scale
    
    # Metriken
    mae = np.mean(np.abs(ideal - quantized))
    rmse = np.sqrt(np.mean((ideal - quantized)**2))
    snr = 20 * np.log10(np.max(np.abs(ideal)) / (rmse + 1e-12))
    effective_bits = np.log2(np.max(np.abs(ideal)) / (mae + 1e-12))
    
    return {
        'ideal': ideal,
        'noisy': quantized,
        'mae': mae,
        'rmse': rmse,
        'snr_db': snr,
        'effective_bits': effective_bits,
        'noise_level': noise_level,
        'drift_factor': drift_factor,
        'crosstalk_factor': crosstalk_factor,
        'bit_precision': bit_precision,
        'params': {'A_shape': A.shape, 'B_shape': B.shape}
    }

In [8]:
# Zelle 3: Batch-Generierung
def generate_batch(n_samples=100, matrix_size=64, **kwargs):
    results = []
    for i in range(n_samples):
        A = np.random.randn(matrix_size, matrix_size) * 0.5
        B = np.random.randn(matrix_size, matrix_size) * 0.5
        result = photonic_matrix_mult(A, B, seed=i, **kwargs)
        result['sample_id'] = i
        results.append(result)
    return results

In [9]:
# Zelle 4: Experimente erzeugen
experiments = []

# Verschiedene Szenarien
experiments.extend(generate_batch(n_samples=80, matrix_size=32, 
                                 noise_level=0.005, drift_factor=0.002))

experiments.extend(generate_batch(n_samples=80, matrix_size=64, 
                                 noise_level=0.015, drift_factor=0.005))

experiments.extend(generate_batch(n_samples=50, matrix_size=128, 
                                 noise_level=0.025, drift_factor=0.008))

print(f"{len(experiments)} synthetische Experimente erzeugt")

210 synthetische Experimente erzeugt


In [10]:
# Zelle 5: In Datenbank speichern
df_list = []
for exp in experiments:
    row = {
        'sample_id': exp['sample_id'],
        'matrix_size': exp['params']['A_shape'][0],
        'noise_level': exp['noise_level'],
        'drift_factor': exp['drift_factor'],
        'crosstalk_factor': exp.get('crosstalk_factor', 0.003),
        'bit_precision': exp['bit_precision'],
        'mae': exp['mae'],
        'rmse': exp['rmse'],
        'snr_db': exp['snr_db'],
        'effective_bits': exp['effective_bits'],
        'timestamp': datetime.now()
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

# In DuckDB speichern
con = duckdb.connect("photonic_benchmark.db")
con.execute("DROP TABLE IF EXISTS experiments")
con.execute("CREATE TABLE experiments AS SELECT * FROM df")

print("✅ Tabelle 'experiments' erfolgreich angelegt und befüllt")
print(f"Anzahl Datensätze: {len(df)}")

✅ Tabelle 'experiments' erfolgreich angelegt und befüllt
Anzahl Datensätze: 210


In [11]:
# Zelle 6: Schnelle Überprüfung
print(con.execute("SELECT matrix_size, COUNT(*) as count, AVG(mae) as avg_mae FROM experiments GROUP BY matrix_size").df())

   matrix_size  count   avg_mae
0           32     80  0.005228
1           64     80  0.019796
2          128     50  0.046145


In [17]:
con.close()